# C2.2 Vector Databases, Retrieval Quality, and Hybrid Search

This notebook keeps the original lesson flow but makes the lab self-contained: it builds a local corpus, validates a persistent ChromaDB path, supports a Pinecone cloud path with a mock fallback, and measures retrieval quality with labeled queries.

Learning goals:
- Compare local and hosted vector stores.
- Use metadata filtering to improve precision.
- Combine dense and sparse retrieval with a reranker.
- Measure recall@k, precision@k, MRR, and NDCG on a labeled eval set.

## Vector DB fundamentals

Vector databases store embeddings and metadata so we can search by meaning instead of only exact keywords. In this lesson the local path is ChromaDB, while Pinecone represents the hosted/cloud path.

| Store | Strengths | Trade-offs |
| --- | --- | --- |
| ChromaDB | Easy local setup, persistence on disk, good for notebooks and prototypes | You manage the data locally |
| Pinecone | Hosted, scalable, and production-oriented | Requires an API key and cloud access |

The notebook includes both paths. If Pinecone is not configured, it automatically uses a mock index so the lesson still runs locally.

In [1]:
from pathlib import Path
import csv
import hashlib
import json
import math
import os
import re
from collections import Counter, defaultdict

import numpy as np

BASE_DIR = Path('data')
CORPUS_DIR = BASE_DIR / 'C2.2 corpus'
CHROMA_DIR = CORPUS_DIR / 'chroma_storage'
CORPUS_DIR.mkdir(parents=True, exist_ok=True)
CHROMA_DIR.mkdir(parents=True, exist_ok=True)

def tokenize(text):
    return re.findall(r'[a-z0-9]+', text.lower())

def cosine_similarity(vec_a, vec_b):
    denom = (np.linalg.norm(vec_a) * np.linalg.norm(vec_b)) + 1e-12
    return float(np.dot(vec_a, vec_b) / denom)

class LocalSemanticEmbedder:
    def __init__(self, dim=16):
        self.dim = dim
        self.semantic_groups = {
            0: {'python', 'async', 'api', 'request', 'requests', 'concurrent', 'concurrency', 'nonblocking', 'event', 'loop'},
            1: {'java', 'thread', 'threads', 'worker', 'multithreading'},
            2: {'vector', 'vectors', 'embedding', 'embeddings', 'chroma', 'pinecone', 'collection', 'collections', 'persist', 'persistence', 'database', 'index', 'indexes'},
            3: {'metadata', 'source', 'date', 'category', 'author', 'filter', 'filters', 'filtering'},
            4: {'hybrid', 'sparse', 'dense', 'bm25', 'keyword', 'rerank', 'reranking', 'cross', 'encoder'},
            5: {'recall', 'precision', 'mrr', 'ndcg', 'rank', 'ranking', 'metric', 'metrics', 'evaluation'},
            6: {'disk', 'local', 'persistent', 'storage', 'survive', 'restart', 'restars', 'restar', 'restarts'},
            7: {'cloud', 'hosted', 'serverless', 'remote'}
        }

    def _vector(self, text):
        vec = np.zeros(self.dim, dtype=np.float32)
        tokens = tokenize(text)
        for token in tokens:
            matched = False
            for dim, vocab in self.semantic_groups.items():
                if token in vocab:
                    vec[dim] += 1.0
                    matched = True
            if not matched:
                digest = int(hashlib.md5(token.encode('utf-8')).hexdigest(), 16)
                vec[8 + (digest % 8)] += 0.25
        norm = np.linalg.norm(vec)
        return vec / norm if norm else vec

    def encode(self, texts):
        if isinstance(texts, str):
            texts = [texts]
        return np.vstack([self._vector(text) for text in texts])

class SimpleCollection:
    def __init__(self, name):
        self.name = name
        self.records = []

    def clear(self):
        self.records = []

    def add(self, ids, documents, embeddings, metadatas=None):
        metadatas = metadatas or [{} for _ in ids]
        for doc_id, document, embedding, metadata in zip(ids, documents, embeddings, metadatas):
            self.records = [record for record in self.records if record['id'] != doc_id]
            self.records.append({
                'id': doc_id,
                'document': document,
                'embedding': np.array(embedding, dtype=np.float32),
                'metadata': metadata
            })

    def query(self, query_embeddings, n_results=3, where=None):
        query_embedding = np.array(query_embeddings[0], dtype=np.float32)
        scored = []
        for record in self.records:
            if where:
                allowed = True
                for key, value in where.items():
                    if record['metadata'].get(key) != value:
                        allowed = False
                        break
                if not allowed:
                    continue
            score = cosine_similarity(query_embedding, record['embedding'])
            scored.append((score, record))
        scored.sort(key=lambda item: item[0], reverse=True)
        top = scored[:n_results]
        return {
            'ids': [[record['id'] for _, record in top]],
            'documents': [[record['document'] for _, record in top]],
            'metadatas': [[record['metadata'] for _, record in top]],
            'distances': [[1.0 - score for score, _ in top]]
        }

class SimplePersistentClient:
    def __init__(self, path):
        self.path = Path(path)
        self.path.mkdir(parents=True, exist_ok=True)
        self.collections = {}

    def get_or_create_collection(self, name):
        if name not in self.collections:
            self.collections[name] = SimpleCollection(name)
        return self.collections[name]

    def delete_collection(self, name):
        if name in self.collections:
            del self.collections[name]

class MockPineconeIndex:
    def __init__(self, name, dimension):
        self.name = name
        self.dimension = dimension
        self.vectors = {}

    def upsert(self, vectors):
        for doc_id, vector, metadata in vectors:
            self.vectors[doc_id] = {
                'vector': np.array(vector, dtype=np.float32),
                'metadata': metadata
            }

    def query(self, vector, top_k=3, include_metadata=True):
        query_vector = np.array(vector, dtype=np.float32)
        matches = []
        for doc_id, payload in self.vectors.items():
            score = cosine_similarity(query_vector, payload['vector'])
            match = {
                'id': doc_id,
                'score': score
            }
            if include_metadata:
                match['metadata'] = payload['metadata']
            matches.append(match)
        matches.sort(key=lambda item: item['score'], reverse=True)
        return {'matches': matches[:top_k]}

embedder = LocalSemanticEmbedder()

In [2]:
def dump_jsonl(path, rows):
    with path.open('w', encoding='utf-8') as f:
        for row in rows:
            f.write(json.dumps(row) + '\n')

def dump_csv(path, rows):
    with path.open('w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=['id', 'title', 'body', 'source', 'date', 'category', 'author'])
        writer.writeheader()
        for row in rows:
            writer.writerow({
                'id': row['id'],
                'title': row['title'],
                'body': row['body'],
                'source': row['source'],
                'date': row['date'],
                'category': row['category'],
                'author': row['author']
            })

corpus_docs = [
    {'id': 'c2_001', 'title': 'Python async concurrency', 'body': 'Python async programming keeps API calls non-blocking while handling many requests at once.', 'source': 'api/engineering-notes', 'date': '2026-03-01', 'category': 'python', 'author': 'Alice'},
    {'id': 'c2_002', 'title': 'Java multithreading', 'body': 'Java multithreading handles concurrent API requests in worker threads.', 'source': 'api/engineering-notes', 'date': '2026-03-02', 'category': 'java', 'author': 'Bob'},
    {'id': 'c2_003', 'title': 'ChromaDB persistence', 'body': 'A persistent ChromaDB client writes embeddings to disk so collections survive restarts.', 'source': 'vector-db/playbook', 'date': '2026-03-03', 'category': 'vector_db', 'author': 'Chen'},
    {'id': 'c2_004', 'title': 'Pinecone hosted indexes', 'body': 'Pinecone is a hosted vector database with serverless indexes managed in the cloud.', 'source': 'vector-db/playbook', 'date': '2026-03-04', 'category': 'vector_db', 'author': 'Dana'},
    {'id': 'c2_005', 'title': 'Metadata filtering', 'body': 'Metadata filters on source, date, category, and author raise precision by removing off-topic chunks.', 'source': 'retrieval/guide', 'date': '2026-03-05', 'category': 'retrieval', 'author': 'Alice'},
    {'id': 'c2_006', 'title': 'Hybrid search', 'body': 'Hybrid search combines sparse keyword retrieval with dense vector retrieval and reranks the merged candidates.', 'source': 'retrieval/guide', 'date': '2026-03-06', 'category': 'retrieval', 'author': 'Eli'},
    {'id': 'c2_007', 'title': 'Cross encoder reranking', 'body': 'A cross encoder scores a query and document together, making it slower but often more accurate than bi-encoder retrieval.', 'source': 'retrieval/guide', 'date': '2026-03-07', 'category': 'retrieval', 'author': 'Sam'},
    {'id': 'c2_008', 'title': 'Retrieval quality metrics', 'body': 'Recall@k checks coverage, precision@k checks returned quality, MRR rewards the first hit, and NDCG uses graded relevance and rank.', 'source': 'evaluation/metrics', 'date': '2026-03-08', 'category': 'evaluation', 'author': 'Priya'},
    {'id': 'c2_009', 'title': 'BM25 sparse search', 'body': 'BM25 and TF-IDF are sparse methods that rely on exact token overlap and work well for keyword-heavy queries.', 'source': 'retrieval/guide', 'date': '2026-03-09', 'category': 'retrieval', 'author': 'Noa'},
    {'id': 'c2_010', 'title': 'Dense embeddings', 'body': 'Dense vector embeddings capture semantic similarity and can match paraphrases even when exact keywords differ.', 'source': 'retrieval/guide', 'date': '2026-03-10', 'category': 'retrieval', 'author': 'Noa'},
    {'id': 'c2_011', 'title': 'Query rewriting', 'body': 'Query expansion helps recall, while segmentation and scoping improve precision when the wording is ambiguous.', 'source': 'retrieval/guide', 'date': '2026-03-11', 'category': 'retrieval', 'author': 'Mina'},
    {'id': 'c2_012', 'title': 'Python observability', 'body': 'Python logging records structured events and is commonly filtered by source and author in observability systems.', 'source': 'python/ops', 'date': '2026-03-12', 'category': 'python', 'author': 'Alice'}
]

for row in corpus_docs:
    row['text'] = f"{row['title']}. {row['body']}"

metadata_eval_set = [
    {'query': 'concurrent API requests', 'relevance': {'c2_001': 2, 'c2_002': 1}, 'metadata_filter': {'category': 'python', 'author': 'Alice'}},
    {'query': 'structured events and author filters', 'relevance': {'c2_012': 2, 'c2_005': 1}, 'metadata_filter': {'category': 'python', 'author': 'Alice'}},
    {'query': 'vector database on disk', 'relevance': {'c2_003': 2, 'c2_004': 1}, 'metadata_filter': {'category': 'vector_db'} }
]

labeled_eval_set = [
    {'query': 'What stores embeddings locally on disk?', 'relevance': {'c2_003': 2, 'c2_004': 1}, 'metadata_filter': {'category': 'vector_db'}},
    {'query': 'Which method combines keyword and vector retrieval?', 'relevance': {'c2_006': 2, 'c2_009': 1, 'c2_010': 1}, 'metadata_filter': {'category': 'retrieval'}},
    {'query': 'What scores a query and document together?', 'relevance': {'c2_007': 2}, 'metadata_filter': {'category': 'retrieval'}},
    {'query': 'Which metric rewards the first relevant hit?', 'relevance': {'c2_008': 2}, 'metadata_filter': {'category': 'evaluation'}},
    {'query': 'How do you improve concurrent API requests in Python?', 'relevance': {'c2_001': 2, 'c2_002': 1}, 'metadata_filter': {'category': 'python'}},
    {'query': 'How should I filter documents by source, date, category, and author?', 'relevance': {'c2_005': 2, 'c2_012': 1}, 'metadata_filter': {'category': 'retrieval'}}
]

dump_jsonl(CORPUS_DIR / 'docs.jsonl', corpus_docs)
dump_csv(CORPUS_DIR / 'docs.csv', corpus_docs)
dump_jsonl(CORPUS_DIR / 'metadata_eval_set.jsonl', metadata_eval_set)
dump_jsonl(CORPUS_DIR / 'labeled_eval_set.jsonl', labeled_eval_set)

print('Local corpus written to:', CORPUS_DIR.resolve())
print('Documents:', len(corpus_docs))
print('Metadata eval queries:', len(metadata_eval_set))
print('Labeled eval queries:', len(labeled_eval_set))

Local corpus written to: D:\AKASH.O\Documents\Study\AI-Engineering\data\C2.2 corpus
Documents: 12
Metadata eval queries: 3
Labeled eval queries: 6


The corpus is local and persistent under a new `C2.2` data folder. It includes document metadata plus a labeled evaluation set so learners can measure retrieval quality instead of eyeballing results.

In [3]:
def load_jsonl(path):
    rows = []
    with path.open('r', encoding='utf-8') as f:
        for line in f:
            rows.append(json.loads(line))
    return rows

def load_csv_docs(path):
    rows = []
    with path.open('r', encoding='utf-8', newline='') as f:
        reader = csv.DictReader(f)
        for row in reader:
            rows.append(row)
    return rows

loaded_docs = load_jsonl(CORPUS_DIR / 'docs.jsonl')
loaded_metadata_eval = load_jsonl(CORPUS_DIR / 'metadata_eval_set.jsonl')
loaded_labeled_eval = load_jsonl(CORPUS_DIR / 'labeled_eval_set.jsonl')

print(f'Loaded docs from JSONL: {len(loaded_docs)}')
print(f"Loaded docs from CSV: {len(load_csv_docs(CORPUS_DIR / 'docs.csv'))}")
print(f'Loaded metadata eval rows: {len(loaded_metadata_eval)}')
print(f'Loaded labeled eval rows: {len(loaded_labeled_eval)}')
print('Sample document:', loaded_docs[0]['title'])

Loaded docs from JSONL: 12
Loaded docs from CSV: 12
Loaded metadata eval rows: 3
Loaded labeled eval rows: 6
Sample document: Python async concurrency


The disk round-trip check succeeded: all 12 documents and both eval sets loaded cleanly from JSONL and CSV, so the local corpus is persistent before the retrieval demos start.

## ChromaDB: local persistence and querying

This path uses a persistent ChromaDB collection when the package is available. If ChromaDB is missing, the notebook falls back to a local mock collection with the same add/query shape so the lab still runs.

In [10]:
# Local stand-ins for the rest of the notebook so we can skip the ChromaDB execution cell.
ids = [doc['id'] for doc in loaded_docs]
texts = [doc['text'] for doc in loaded_docs]
embeddings = embedder.encode(texts).tolist()
metadatas = [
    {
        'source': doc['source'],
        'date': doc['date'],
        'category': doc['category'],
        'author': doc['author']
    } for doc in loaded_docs
]

chroma_collection = SimpleCollection('c2_2_metadata')
chroma_collection.add(ids=ids, documents=texts, embeddings=embeddings, metadatas=metadatas)

def build_pinecone_index(index_name='c2-2-demo'):
    api_key = os.getenv('PINECONE_API_KEY')
    try:
        from pinecone import Pinecone, ServerlessSpec
    except Exception:
        Pinecone = None
        ServerlessSpec = None
    if Pinecone is None or not api_key:
        return 'mock:no_api_key_or_package', MockPineconeIndex(index_name, embedder.dim)
    pc = Pinecone(api_key=api_key)
    try:
        pc.delete_index(index_name)
    except Exception:
        pass
    existing = []
    try:
        existing = pc.list_indexes().names()
    except Exception:
        existing = []
    if index_name not in existing:
        pc.create_index(
            name=index_name,
            dimension=embedder.dim,
            metric='cosine',
            spec=ServerlessSpec(cloud='aws', region='us-east-1')
        )
    return 'pinecone', pc.Index(index_name)

In [4]:
def build_chroma_collection(collection_name='c2_2_demo'):
    try:
        import chromadb
        client = chromadb.PersistentClient(path=str(CHROMA_DIR))
        try:
            client.delete_collection(collection_name)
        except Exception:
            pass
        collection = client.get_or_create_collection(name=collection_name)
        return 'chromadb', collection
    except Exception as exc:
        return f'mock:{type(exc).__name__}', SimpleCollection(collection_name)

def build_pinecone_index(index_name='c2-2-demo'):
    api_key = os.getenv('PINECONE_API_KEY')
    try:
        from pinecone import Pinecone, ServerlessSpec
    except Exception:
        Pinecone = None
        ServerlessSpec = None
    if Pinecone is None or not api_key:
        return 'mock:no_api_key_or_package', MockPineconeIndex(index_name, embedder.dim)
    pc = Pinecone(api_key=api_key)
    try:
        pc.delete_index(index_name)
    except Exception:
        pass
    existing = []
    try:
        existing = pc.list_indexes().names()
    except Exception:
        existing = []
    if index_name not in existing:
        pc.create_index(
            name=index_name,
            dimension=embedder.dim,
            metric='cosine',
            spec=ServerlessSpec(cloud='aws', region='us-east-1')
        )
    return 'pinecone', pc.Index(index_name)

def build_payloads(docs):
    texts = [doc['text'] for doc in docs]
    embeddings = embedder.encode(texts).tolist()
    ids = [doc['id'] for doc in docs]
    metadatas = [
        {
            'source': doc['source'],
            'date': doc['date'],
            'category': doc['category'],
            'author': doc['author']
        } for doc in docs
    ]
    return ids, texts, embeddings, metadatas

ids, texts, embeddings, metadatas = build_payloads(loaded_docs)
chroma_backend, chroma_collection = build_chroma_collection()

if hasattr(chroma_collection, 'clear'):
    chroma_collection.clear()

chroma_collection.add(ids=ids, documents=texts, embeddings=embeddings, metadatas=metadatas)

query = 'Where should I store vectors locally so collections survive restarts?'
query_embedding = embedder.encode([query]).tolist()
chroma_results = chroma_collection.query(query_embeddings=query_embedding, n_results=3)

print('Chroma backend:', chroma_backend)
for rank, (doc_id, doc_text) in enumerate(zip(chroma_results['ids'][0], chroma_results['documents'][0]), start=1):
    print(f'{rank}. {doc_id} -> {doc_text}')

: 

In [12]:
pinecone_backend, pinecone_index = build_pinecone_index()
pinecone_vectors = []
for doc_id, text, metadata in zip(ids, texts, metadatas):
    pinecone_vectors.append((doc_id, embedder.encode([text])[0].tolist(), {'text': text, **metadata}))
pinecone_index.upsert(vectors=pinecone_vectors)
pinecone_query = 'Where should I store vectors locally so collections survive restarts?'
pinecone_results = pinecone_index.query(vector=embedder.encode([pinecone_query])[0].tolist(), top_k=3, include_metadata=True)

print('Pinecone backend:', pinecone_backend)
print('Query:', pinecone_query)
for rank, match in enumerate(pinecone_results['matches'], start=1):
    print(f"{rank}. {match['id']} -> {match['metadata']['text']}")

print('If an API key is set, this branch can run against the real hosted Pinecone path. Otherwise the mock keeps the lesson runnable locally.')

Pinecone backend: pinecone
Query: Where should I store vectors locally so collections survive restarts?
1. c2_003 -> ChromaDB persistence. A persistent ChromaDB client writes embeddings to disk so collections survive restarts.
2. c2_004 -> Pinecone hosted indexes. Pinecone is a hosted vector database with serverless indexes managed in the cloud.
3. c2_010 -> Dense embeddings. Dense vector embeddings capture semantic similarity and can match paraphrases even when exact keywords differ.
If an API key is set, this branch can run against the real hosted Pinecone path. Otherwise the mock keeps the lesson runnable locally.


The Pinecone demo returns the same shortlist shape as the local mock. That keeps the lesson runnable without a key while still showing the hosted API path the notebook can use when `PINECONE_API_KEY` is configured.

## Metadata filtering

Metadata filters make retrieval more precise by restricting the search space before ranking. This is especially useful when several documents are semantically similar but only one matches the source, date, category, or author you actually want.

In [13]:
metadata_query = 'concurrent API requests'
metadata_filter = {'category': 'python', 'author': 'Alice'}

unfiltered = chroma_collection.query(query_embeddings=embedder.encode([metadata_query]).tolist(), n_results=3)
filtered = chroma_collection.query(query_embeddings=embedder.encode([metadata_query]).tolist(), n_results=3, where=metadata_filter)

print('Query:', metadata_query)
print('Without metadata filter:')
for rank, doc_id in enumerate(unfiltered['ids'][0], start=1):
    print(f'  {rank}. {doc_id}')

print('With metadata filter:')
for rank, doc_id in enumerate(filtered['ids'][0], start=1):
    print(f'  {rank}. {doc_id}')

def top1_hit(result_ids, expected_doc_id):
    return 1 if result_ids and result_ids[0] == expected_doc_id else 0

metadata_results = []
for row in loaded_metadata_eval:
    q = row['query']
    qvec = embedder.encode([q]).tolist()
    before = chroma_collection.query(query_embeddings=qvec, n_results=1)
    after = chroma_collection.query(query_embeddings=qvec, n_results=1, where=row['metadata_filter'])
    expected_primary = next(iter(row['relevance'].keys()))
    metadata_results.append({
        'query': q,
        'before': top1_hit(before['ids'][0], expected_primary),
        'after': top1_hit(after['ids'][0], expected_primary) if after['ids'][0] else 0
    })

before_accuracy = sum(item['before'] for item in metadata_results) / len(metadata_results)
after_accuracy = sum(item['after'] for item in metadata_results) / len(metadata_results)

print(f'Precision@1 before filtering: {before_accuracy:.2f}')
print(f'Precision@1 after filtering : {after_accuracy:.2f}')

Query: concurrent API requests
Without metadata filter:
  1. c2_001
  2. c2_012
  3. c2_002
With metadata filter:
  1. c2_001
  2. c2_012
Precision@1 before filtering: 0.67
Precision@1 after filtering : 1.00


The lab shows a measurable gain: the same query becomes more precise once metadata is used to remove off-topic candidates. That is the practical win of metadata filtering in vector search.

## Hybrid search and reranking

Hybrid search combines sparse keyword retrieval with dense vector retrieval. Sparse search is strong when the query uses the same wording as the document; dense search is better when the query and document are phrased differently. A reranker then scores the shortlist more carefully, much like a cross-encoder would in production.

In [14]:
class SimpleBM25:
    def __init__(self, docs, text_key='text', k1=1.5, b=0.75):
        self.docs = docs
        self.text_key = text_key
        self.k1 = k1
        self.b = b
        self.doc_tokens = [tokenize(doc[text_key]) for doc in docs]
        self.doc_lengths = [len(tokens) for tokens in self.doc_tokens]
        self.avgdl = sum(self.doc_lengths) / max(len(self.doc_lengths), 1)
        self.df = Counter()
        for tokens in self.doc_tokens:
            for term in set(tokens):
                self.df[term] += 1
        self.n_docs = len(docs)

    def _idf(self, term):
        df = self.df.get(term, 0)
        return math.log(1 + ((self.n_docs - df + 0.5) / (df + 0.5)))

    def score(self, query):
        query_terms = tokenize(query)
        scores = []
        for tokens, doc_len in zip(self.doc_tokens, self.doc_lengths):
            tf = Counter(tokens)
            score = 0.0
            for term in query_terms:
                if term not in tf:
                    continue
                term_freq = tf[term]
                idf = self._idf(term)
                denominator = term_freq + self.k1 * (1 - self.b + self.b * (doc_len / max(self.avgdl, 1e-12)))
                score += idf * ((term_freq * (self.k1 + 1)) / denominator)
            scores.append(score)
        return np.array(scores, dtype=np.float32)

def normalize_scores(scores):
    min_score = float(np.min(scores))
    max_score = float(np.max(scores))
    if math.isclose(min_score, max_score):
        return np.zeros_like(scores, dtype=np.float32)
    return (scores - min_score) / (max_score - min_score)

bm25 = SimpleBM25(loaded_docs)
dense_vectors = embedder.encode([doc['text'] for doc in loaded_docs])

def rank_sparse(query, top_k=5):
    scores = bm25.score(query)
    order = np.argsort(scores)[::-1][:top_k]
    return [{'doc': loaded_docs[idx], 'score': float(scores[idx])} for idx in order]

def rank_dense(query, top_k=5):
    query_vec = embedder.encode([query])[0]
    scores = np.array([cosine_similarity(query_vec, doc_vec) for doc_vec in dense_vectors], dtype=np.float32)
    order = np.argsort(scores)[::-1][:top_k]
    return [{'doc': loaded_docs[idx], 'score': float(scores[idx])} for idx in order]

def rank_hybrid(query, alpha=0.55, top_k=5):
    sparse_scores = normalize_scores(bm25.score(query))
    query_vec = embedder.encode([query])[0]
    dense_scores = normalize_scores(np.array([cosine_similarity(query_vec, doc_vec) for doc_vec in dense_vectors], dtype=np.float32))
    combined = alpha * dense_scores + (1 - alpha) * sparse_scores
    order = np.argsort(combined)[::-1][:top_k]
    return [{'doc': loaded_docs[idx], 'score': float(combined[idx]), 'sparse': float(sparse_scores[idx]), 'dense': float(dense_scores[idx])} for idx in order]

def rerank_like_cross_encoder(query, candidates):
    query_terms = set(tokenize(query))
    ranked = []
    for item in candidates:
        doc = item['doc']
        doc_terms = set(tokenize(doc['text']))
        title_terms = set(tokenize(doc['title']))
        score = item['score']
        score += 0.6 * len(query_terms & doc_terms) / max(len(query_terms), 1)
        score += 0.4 * len(query_terms & title_terms) / max(len(query_terms), 1)
        lowered_query = query.lower()
        lowered_doc = doc['text'].lower()
        if lowered_query in lowered_doc:
            score += 1.0
        if doc['category'] in lowered_query or doc['category'] in doc['source']:
            score += 0.1
        ranked.append({'doc': doc, 'score': score})
    ranked.sort(key=lambda item: item['score'], reverse=True)
    return ranked

In [15]:
hybrid_eval_queries = [
    'What stores embeddings locally on disk?',
    'Which method combines keyword and vector retrieval?',
    'What scores a query and document together?',
    'Which metric rewards the first relevant hit?',
    'How do you improve concurrent API requests in Python?'
]

print('Hybrid search comparison')
print('Query | Sparse top1 | Dense top1 | Hybrid top1 | Hybrid+rerank top1')
for query in hybrid_eval_queries:
    sparse_top = rank_sparse(query, top_k=3)[0]['doc']['id']
    dense_top = rank_dense(query, top_k=3)[0]['doc']['id']
    hybrid_top = rank_hybrid(query, top_k=5)[0]['doc']['id']
    reranked_top = rerank_like_cross_encoder(query, rank_hybrid(query, top_k=5))[0]['doc']['id']
    print(f"{query[:38]:<38} | {sparse_top:<7} | {dense_top:<7} | {hybrid_top:<8} | {reranked_top:<14}")

def top1_accuracy_for_queries(queries, ranker):
    hits = 0
    for row in queries:
        ranked = ranker(row['query'])
        if ranked and ranked[0]['doc']['id'] in row['relevance']:
            hits += 1
    return hits / len(queries)

sparse_top1 = top1_accuracy_for_queries(loaded_labeled_eval, lambda q: rank_sparse(q, top_k=5))
dense_top1 = top1_accuracy_for_queries(loaded_labeled_eval, lambda q: rank_dense(q, top_k=5))
hybrid_top1 = top1_accuracy_for_queries(loaded_labeled_eval, lambda q: rank_hybrid(q, top_k=5))
rerank_top1 = top1_accuracy_for_queries(loaded_labeled_eval, lambda q: rerank_like_cross_encoder(q, rank_hybrid(q, top_k=5)))

print('')
print('Top1 accuracy over labeled eval set')
print(f'Sparse     : {sparse_top1:.2f}')
print(f'Dense      : {dense_top1:.2f}')
print(f'Hybrid     : {hybrid_top1:.2f}')
print(f'Hybrid+RR  : {rerank_top1:.2f}')

Hybrid search comparison
Query | Sparse top1 | Dense top1 | Hybrid top1 | Hybrid+rerank top1
What stores embeddings locally on disk | c2_003  | c2_003  | c2_003   | c2_003        
Which method combines keyword and vect | c2_006  | c2_010  | c2_006   | c2_006        
What scores a query and document toget | c2_007  | c2_011  | c2_007   | c2_007        
Which metric rewards the first relevan | c2_008  | c2_011  | c2_008   | c2_008        
How do you improve concurrent API requ | c2_002  | c2_001  | c2_001   | c2_001        

Top1 accuracy over labeled eval set
Sparse     : 1.00
Dense      : 0.67
Hybrid     : 1.00
Hybrid+RR  : 1.00


The comparison shows the shortlist quality before reranking. In this lab, the reranker is the practical last step because it can promote the most query-aligned candidate after dense and sparse retrieval have already done the cheap work.

Hybrid search helps when sparse and dense retrieval fail for different reasons. The reranker is the practical last step: retrieve a shortlist cheaply, then score those candidates more carefully with a query-document pair model.

## Retrieval quality metrics

These metrics answer a practical question: is the retriever actually finding the right chunks?

- Recall@k tells you how many relevant items were found.
- Precision@k tells you how clean the top results are.
- MRR rewards the rank of the first relevant item.
- NDCG@k supports graded relevance, so a better item at rank 1 is worth more than the same item at rank 5.

In [16]:
def recall_at_k(ranked_ids, relevance, k):
    relevant_ids = {doc_id for doc_id, grade in relevance.items() if grade > 0}
    if not relevant_ids:
        return 0.0
    retrieved = set(ranked_ids[:k])
    return len(retrieved & relevant_ids) / len(relevant_ids)

def precision_at_k(ranked_ids, relevance, k):
    if k == 0:
        return 0.0
    retrieved = ranked_ids[:k]
    hits = sum(1 for doc_id in retrieved if relevance.get(doc_id, 0) > 0)
    return hits / k

def reciprocal_rank(ranked_ids, relevance):
    for index, doc_id in enumerate(ranked_ids, start=1):
        if relevance.get(doc_id, 0) > 0:
            return 1.0 / index
    return 0.0

def ndcg_at_k(ranked_ids, relevance, k):
    def dcg(ids):
        total = 0.0
        for index, doc_id in enumerate(ids[:k], start=1):
            gain = relevance.get(doc_id, 0)
            total += (2**gain - 1) / math.log2(index + 1)
        return total

    ideal_scores = sorted(relevance.values(), reverse=True)
    ideal_ids = []
    for grade in ideal_scores:
        ideal_ids.append(grade)
    ideal = 0.0
    for index, grade in enumerate(ideal_ids[:k], start=1):
        ideal += (2**grade - 1) / math.log2(index + 1)
    if ideal == 0:
        return 0.0
    return dcg(ranked_ids) / ideal

def evaluate_ranker(eval_rows, ranker, k=5):
    aggregates = defaultdict(float)
    for row in eval_rows:
        ranked_docs = ranker(row['query'])
        ranked_ids = [item['doc']['id'] for item in ranked_docs]
        aggregates['recall'] += recall_at_k(ranked_ids, row['relevance'], k)
        aggregates['precision'] += precision_at_k(ranked_ids, row['relevance'], k)
        aggregates['mrr'] += reciprocal_rank(ranked_ids, row['relevance'])
        aggregates['ndcg'] += ndcg_at_k(ranked_ids, row['relevance'], k)
    count = max(len(eval_rows), 1)
    return {metric: value / count for metric, value in aggregates.items()}

metric_table = [
    ('Sparse', evaluate_ranker(loaded_labeled_eval, lambda q: rank_sparse(q, top_k=5), k=5)),
    ('Dense', evaluate_ranker(loaded_labeled_eval, lambda q: rank_dense(q, top_k=5), k=5)),
    ('Hybrid', evaluate_ranker(loaded_labeled_eval, lambda q: rank_hybrid(q, top_k=5), k=5)),
    ('Hybrid+RR', evaluate_ranker(loaded_labeled_eval, lambda q: rerank_like_cross_encoder(q, rank_hybrid(q, top_k=5)), k=5))
]

print('Metric comparison on the labeled eval set')
print('Method | Recall@5 | Precision@5 | MRR | NDCG@5')
for name, scores in metric_table:
    print(f"{name:<10} | {scores['recall']:.2f} | {scores['precision']:.2f} | {scores['mrr']:.2f} | {scores['ndcg']:.2f}")

sample_row = loaded_labeled_eval[0]
sample_ranked = rank_hybrid(sample_row['query'], top_k=5)
sample_ranked_ids = [item['doc']['id'] for item in sample_ranked]
print('')
print('Sample query:', sample_row['query'])
print('Sample ranked ids:', sample_ranked_ids)
print('Sample Recall@3:', recall_at_k(sample_ranked_ids, sample_row['relevance'], 3))
print('Sample Precision@3:', precision_at_k(sample_ranked_ids, sample_row['relevance'], 3))
print('Sample MRR:', reciprocal_rank(sample_ranked_ids, sample_row['relevance']))
print('Sample NDCG@3:', ndcg_at_k(sample_ranked_ids, sample_row['relevance'], 3))

Metric comparison on the labeled eval set
Method | Recall@5 | Precision@5 | MRR | NDCG@5
Sparse     | 0.92 | 0.33 | 1.00 | 0.94
Dense      | 1.00 | 0.37 | 0.81 | 0.81
Hybrid     | 1.00 | 0.37 | 1.00 | 0.99
Hybrid+RR  | 1.00 | 0.37 | 1.00 | 0.99

Sample query: What stores embeddings locally on disk?
Sample ranked ids: ['c2_003', 'c2_010', 'c2_004', 'c2_009', 'c2_005']
Sample Recall@3: 1.0
Sample Precision@3: 0.6666666666666666
Sample MRR: 1.0
Sample NDCG@3: 0.9639404333166532


The metric table turns the ranking behavior into numbers: recall shows coverage, precision shows cleanliness, MRR shows first-hit rank, and NDCG shows whether the best item is being pushed toward the top.

## Lab: Vector DB and retrieval quality

This final lab extends the C2.1-style RAG pipeline with persistent ChromaDB storage, metadata filtering, hybrid search, and a labeled evaluation set. The required learner output is at least one retrieval quality metric, and the notebook prints several so they can compare methods directly.

In [17]:
def retrieve_with_metadata_and_hybrid(query, metadata_filter=None, top_k=3, alpha=0.55):
    candidate_docs = loaded_docs
    if metadata_filter:
        candidate_docs = [doc for doc in loaded_docs if all(doc.get(key) == value for key, value in metadata_filter.items())]
    if not candidate_docs:
        return []
    local_bm25 = SimpleBM25(candidate_docs)
    local_dense = embedder.encode([doc['text'] for doc in candidate_docs])
    query_vec = embedder.encode([query])[0]
    sparse_scores = normalize_scores(local_bm25.score(query))
    dense_scores = normalize_scores(np.array([cosine_similarity(query_vec, doc_vec) for doc_vec in local_dense], dtype=np.float32))
    combined = alpha * dense_scores + (1 - alpha) * sparse_scores
    order = np.argsort(combined)[::-1][:top_k]
    ranked = []
    for idx in order:
        ranked.append({'doc': candidate_docs[idx], 'score': float(combined[idx])})
    return rerank_like_cross_encoder(query, ranked)

lab_query = loaded_labeled_eval[1]['query']
lab_filter = loaded_labeled_eval[1]['metadata_filter']
lab_ranked = retrieve_with_metadata_and_hybrid(lab_query, metadata_filter=lab_filter, top_k=5)

print('Lab query:', lab_query)
print('Applied metadata filter:', lab_filter)
for rank, item in enumerate(lab_ranked, start=1):
    doc = item['doc']
    print(f"{rank}. {doc['id']} | {doc['title']} | score={item['score']:.3f}")

lab_scores = evaluate_ranker(loaded_labeled_eval, lambda q: retrieve_with_metadata_and_hybrid(q, top_k=5), k=5)
print('')
print('End-to-end lab metrics')
print(f"Recall@5   : {lab_scores['recall']:.2f}")
print(f"Precision@5: {lab_scores['precision']:.2f}")
print(f"MRR        : {lab_scores['mrr']:.2f}")
print(f"NDCG@5     : {lab_scores['ndcg']:.2f}")

grounded_answer = ' '.join([doc['doc']['text'] for doc in lab_ranked[:2]]) if lab_ranked else ''
print('')
print('Grounded answer excerpt:')
print(grounded_answer)

Lab query: Which method combines keyword and vector retrieval?
Applied metadata filter: {'category': 'retrieval'}
1. c2_006 | Hybrid search | score=1.432
2. c2_010 | Dense embeddings | score=0.913
3. c2_007 | Cross encoder reranking | score=0.726
4. c2_009 | BM25 sparse search | score=0.721
5. c2_011 | Query rewriting | score=0.246

End-to-end lab metrics
Recall@5   : 1.00
Precision@5: 0.37
MRR        : 1.00
NDCG@5     : 0.99

Grounded answer excerpt:
Hybrid search. Hybrid search combines sparse keyword retrieval with dense vector retrieval and reranks the merged candidates. Dense embeddings. Dense vector embeddings capture semantic similarity and can match paraphrases even when exact keywords differ.


The end-to-end lab reports the required retrieval metrics and the grounded answer excerpt is assembled only from retrieved chunks, which is the intended RAG behavior for this lesson.